# 05 — External validation

NBER recession overlap and VIX>20 baseline on named Stressful labels.

**Section A:** committed CSVs (GMM pointwise; HMM global Viterbi — look-ahead).  
**Section B:** refit on full-sample std; HMM **forward-filtered** (causal) decode — fixes decode look-ahead.  
**Section C:** rolling/expanding causal std + HMM filtered — every step deployable.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from regime_utils import *

raw = load_raw_features()
gmm = load_regime_labels("gmm_regimes.csv")
hmm = load_regime_labels("hmm_regimes.csv")

df = raw.copy()
df["Regime_GMM"] = gmm["Regime_label"]
df["Regime_HMM"] = hmm["Regime_label"]
df["NBER_recession"] = df.index.map(in_recession)
print(f"Regime_GMM non-null: {df['Regime_GMM'].notna().sum()} / {len(df)}")
df.head()


Regime_GMM non-null: 1564 / 1564


,SP500,SP500_Return,VIX,Yield_Spread,Regime_GMM,Regime_HMM,NBER_recession
Date,,,,,,,
1994-01-14,474.910004,0.010605,11.15,1.61,Calm,Transitional,False
1994-01-21,474.720001,-0.000400,11.09,1.64,Calm,Transitional,False
1994-01-28,478.700012,0.008349,9.94,1.60,Calm,Transitional,False
1994-02-04,469.809998,-0.018746,15.25,1.52,Calm,Transitional,False
1994-02-11,470.179993,0.000787,14.46,1.44,Calm,Transitional,False


### A. Committed look-ahead labels

GMM is already pointwise (no decode step). Committed HMM labels use global Viterbi (`hmm.predict`).

In [2]:
nber_table = pd.DataFrame({
    "GMM": nber_stress_rates(df, "Regime_GMM"),
    "HMM (Viterbi)": nber_stress_rates(df, "Regime_HMM"),
})
print(nber_table.round(3))


                                       GMM  HMM (Viterbi)
Recession weeks (% Stressful)        0.522          0.597
Non-recession weeks (% Stressful)    0.093          0.222
Recession weeks (n)                134.000        134.000


In [3]:

for start, end, name in NBER_RECESSIONS:
    sub = df.loc[start:end]
    print(f"{name}: GMM {(sub['Regime_GMM']==STRESS_LABEL).mean():.1%} Stressful, "
          f"HMM {(sub['Regime_HMM']==STRESS_LABEL).mean():.1%} Stressful")


Dot-com: GMM 30.0% Stressful, HMM 57.5% Stressful
GFC: GMM 65.9% Stressful, HMM 58.5% Stressful
COVID: GMM 33.3% Stressful, HMM 75.0% Stressful


In [4]:
baseline_table = pd.DataFrame({
    "GMM": vix_baseline_metrics(df, "Regime_GMM"),
    "HMM (Viterbi)": vix_baseline_metrics(df, "Regime_HMM"),
})
print(baseline_table.round(3))


                                      GMM  HMM (Viterbi)
Agreement with VIX>20 (%)           0.729          0.783
Cohen's kappa (Stress vs rest)      0.350          0.513
Model Stress recall vs baseline     0.318          0.549
Model Stress precision vs baseline  0.951          0.837


### B. Causal filtered decode

Refit on the same full-sample std as notebook 03, then compare GMM (pointwise), HMM Viterbi (look-ahead), and HMM forward-filtered (`filtered_decode` — causal). Names assigned per fit via mean VIX.

In [5]:
std = load_full_sample_std()
X = std.values
raw_fit = load_raw_features().loc[std.index]

gmm = fit_gmm(X)
hmm = fit_hmm(X)
gmm_labels = build_regime_frame(raw_fit, gmm.predict(X), "Regime_GMM")["Regime_label"]
vit_labels = build_regime_frame(raw_fit, viterbi_decode(hmm, X), "Regime_HMM")["Regime_label"]
filt_labels = build_regime_frame(raw_fit, filtered_decode(hmm, X), "Regime_HMM_filtered")["Regime_label"]

df_causal = raw_fit.copy()
df_causal["Regime_GMM"] = gmm_labels.values
df_causal["Regime_HMM_Viterbi"] = vit_labels.values
df_causal["Regime_HMM_filtered"] = filt_labels.values
df_causal["NBER_recession"] = df_causal.index.map(in_recession)

n_diff = (df_causal["Regime_HMM_Viterbi"] != df_causal["Regime_HMM_filtered"]).sum()
print(f"Weeks where Viterbi ≠ filtered: {n_diff} / {len(df_causal)} ({n_diff / len(df_causal):.1%})")

Weeks where Viterbi ≠ filtered: 86 / 1564 (5.5%)


In [6]:
nber_causal = pd.DataFrame({
    "GMM (pointwise)": nber_stress_rates(df_causal, "Regime_GMM"),
    "HMM Viterbi": nber_stress_rates(df_causal, "Regime_HMM_Viterbi"),
    "HMM filtered": nber_stress_rates(df_causal, "Regime_HMM_filtered"),
})
print(nber_causal.round(3))

                                   GMM (pointwise)  HMM Viterbi  HMM filtered
Recession weeks (% Stressful)                0.522        0.597         0.619
Non-recession weeks (% Stressful)            0.093        0.222         0.243
Recession weeks (n)                        134.000      134.000       134.000


In [7]:
for start, end, name in NBER_RECESSIONS:
    sub = df_causal.loc[start:end]
    print(
        f"{name}: GMM {(sub['Regime_GMM'] == STRESS_LABEL).mean():.1%} Stressful, "
        f"Viterbi {(sub['Regime_HMM_Viterbi'] == STRESS_LABEL).mean():.1%}, "
        f"filtered {(sub['Regime_HMM_filtered'] == STRESS_LABEL).mean():.1%} Stressful"
    )

Dot-com: GMM 30.0% Stressful, Viterbi 57.5%, filtered 60.0% Stressful
GFC: GMM 65.9% Stressful, Viterbi 58.5%, filtered 61.0% Stressful
COVID: GMM 33.3% Stressful, Viterbi 75.0%, filtered 75.0% Stressful


In [8]:
vix_causal = pd.DataFrame({
    "GMM (pointwise)": vix_baseline_metrics(df_causal, "Regime_GMM"),
    "HMM Viterbi": vix_baseline_metrics(df_causal, "Regime_HMM_Viterbi"),
    "HMM filtered": vix_baseline_metrics(df_causal, "Regime_HMM_filtered"),
})
print(vix_causal.round(3))

                                    GMM (pointwise)  HMM Viterbi  HMM filtered
Agreement with VIX>20 (%)                     0.729        0.783         0.801
Cohen's kappa (Stress vs rest)                0.350        0.513         0.556
Model Stress recall vs baseline               0.318        0.549         0.598
Model Stress precision vs baseline            0.951        0.837         0.842


In [9]:
vit_stress = df_causal["Regime_HMM_Viterbi"] == STRESS_LABEL
filt_stress = df_causal["Regime_HMM_filtered"] == STRESS_LABEL
print(f"HMM Stress label agreement (Viterbi vs filtered): {(vit_stress == filt_stress).mean():.1%}")
print(f"Filtered recall vs Viterbi Stress: {(filt_stress & vit_stress).sum() / vit_stress.sum():.3f}")
print(f"Filtered precision vs Viterbi Stress: {(filt_stress & vit_stress).sum() / filt_stress.sum():.3f}")

HMM Stress label agreement (Viterbi vs filtered): 95.1%
Filtered recall vs Viterbi Stress: 0.945
Filtered precision vs Viterbi Stress: 0.872


### C. Fully causal variants (causal std + filtered decode)

Section B removes decode look-ahead but still fits on full-sample z-scores (future mean/std). Refit on **rolling** and **expanding** causal standardizations with HMM forward-filtered decode. Rolling/expanding samples are 1,513 weeks (first 51 dropped for `min_periods=52`).

In [10]:
def fit_causal_frame(std: pd.DataFrame) -> pd.DataFrame:
    X = std.values
    raw_fit = load_raw_features().loc[std.index]
    gmm = fit_gmm(X)
    hmm = fit_hmm(X)
    gmm_labels = build_regime_frame(raw_fit, gmm.predict(X), "Regime_GMM")["Regime_label"]
    filt_labels = build_regime_frame(
        raw_fit, filtered_decode(hmm, X), "Regime_HMM_filtered"
    )["Regime_label"]
    out = raw_fit.copy()
    out["Regime_GMM"] = gmm_labels.values
    out["Regime_HMM_filtered"] = filt_labels.values
    out["NBER_recession"] = out.index.map(in_recession)
    return out


causal_variants = {
    "Full-sample": df_causal,
    "Rolling": fit_causal_frame(load_rolling_std()),
    "Expanding": fit_causal_frame(load_expanding_std()),
}
for name, frame in causal_variants.items():
    print(f"{name}: {len(frame)} weeks")

Full-sample: 1564 weeks
Rolling: 1513 weeks
Expanding: 1513 weeks


In [11]:
nber_fully_causal = pd.DataFrame(
    {
        name: nber_stress_rates(frame, "Regime_HMM_filtered")
        for name, frame in causal_variants.items()
    }
)
print("HMM filtered — NBER overlap by std variant:")
print(nber_fully_causal.round(3))

HMM filtered — NBER overlap by std variant:
                                   Full-sample  Rolling  Expanding
Recession weeks (% Stressful)            0.619    0.612      0.634
Non-recession weeks (% Stressful)        0.243    0.292      0.334
Recession weeks (n)                    134.000  134.000    134.000


In [12]:
for start, end, name in NBER_RECESSIONS:
    shares = [
        f"{variant} {(frame.loc[start:end]['Regime_HMM_filtered'] == STRESS_LABEL).mean():.1%}"
        for variant, frame in causal_variants.items()
    ]
    print(f"{name}: {', '.join(shares)}")

Dot-com: Full-sample 60.0%, Rolling 52.5%, Expanding 47.5%
GFC: Full-sample 61.0%, Rolling 59.8%, Expanding 69.5%
COVID: Full-sample 75.0%, Rolling 100.0%, Expanding 75.0%


In [13]:
vix_fully_causal = pd.DataFrame(
    {
        name: vix_baseline_metrics(frame, "Regime_HMM_filtered")
        for name, frame in causal_variants.items()
    }
)
print("HMM filtered — VIX>20 baseline by std variant:")
print(vix_fully_causal.round(3))

HMM filtered — VIX>20 baseline by std variant:
                                    Full-sample  Rolling  Expanding
Agreement with VIX>20 (%)                 0.801    0.695      0.796
Cohen's kappa (Stress vs rest)            0.556    0.342      0.569
Model Stress recall vs baseline           0.598    0.518      0.696
Model Stress precision vs baseline        0.842    0.649      0.773


### D. Regime profile (feature means + drawdown)

Pipeline-generated version of the README's "feature profiles" and "max drawdown within regime" tables. Two drawdown statistics on the **committed** (look-ahead) labels, since spell drawdown is only meaningful for a contiguous path:

- **Max DD (within spell):** worst peak-to-trough decline computed *inside* a single contiguous run of one label — resets at every label change.
- **Mkt DD while in regime:** deepest the whole-market wealth curve is underwater (vs. its all-time-to-date peak) during weeks carrying that label — does not reset at label changes, so it can be large for a Calm week that follows a crash.

In [14]:
def regime_profile(df, label_col):
    g = df.groupby(label_col)
    prof = pd.DataFrame({
        "VIX": g["VIX"].mean(),
        "Yield_Spread": g["Yield_Spread"].mean(),
        "Mean weekly return": g["SP500_Return"].mean(),
        "Share": g.size() / len(df),
        "Max DD (within spell)": max_drawdown_by_spell(df, label_col),
        "Mkt DD while in regime": conditional_drawdown(df, label_col),
    })
    return prof.loc[["Calm", "Transitional", "Stressful"]]

for name, csv in [("GMM", "gmm_regimes.csv"), ("HMM", "hmm_regimes.csv")]:
    print(name)
    print(regime_profile(load_regime_labels(csv), "Regime_label").round(3))


GMM


                 VIX  Yield_Spread  Mean weekly return  Share  \
Calm          14.824         1.130               0.004  0.575   
Transitional  23.759         0.285              -0.001  0.295   
Stressful     31.821         2.086              -0.001  0.130   

              Max DD (within spell)  Mkt DD while in regime  
Calm                         -0.075                  -0.412  
Transitional                 -0.258                  -0.289  
Stressful                    -0.456                  -0.562  
HMM
                 VIX  Yield_Spread  Mean weekly return  Share  \
Calm          15.902         0.290               0.003  0.300   
Transitional  17.321         1.741               0.002  0.446   
Stressful     28.200         0.559              -0.001  0.254   

              Max DD (within spell)  Mkt DD while in regime  
Calm                         -0.101                  -0.252  
Transitional                 -0.156                  -0.437  
Stressful                    -0.456     

In [15]:
# Sanity check: which week drives GMM-Calm's -41.2% "Mkt DD while in regime"?
df_gmm = load_regime_labels("gmm_regimes.csv")
wealth = np.exp(df_gmm["SP500_Return"].cumsum())
dd = (wealth - wealth.cummax()) / wealth.cummax()
print("GMM-Calm underwater minimum at:", dd[df_gmm["Regime_label"] == "Calm"].idxmin().date())
print("Whole-sample underwater minimum at:", dd.idxmin().date())


GMM-Calm underwater minimum at: 2003-04-25
Whole-sample underwater minimum at: 2009-03-06
